# InfraVision: train EDSR x2 super-resolution on POP (Kaggle)

Thermal SR trained on POP train flights; val = POP val flights (no flight in both). Pairs are made on the fly:
clean crop vs blurred / downscaled / re-upscaled / noisy / JPEG copy. The log prints **bicubic's val PSNR**
first: EDSR is only useful if its val PSNR ends up **above** that line.

**One-time setup**
1. Kaggle account → **Settings → Phone verification** (needed for GPU and internet).
2. **Datasets → New Dataset** → upload `pop_raw.zip` (private). Any name is fine.
3. Here: **Add Input** → that dataset. **Session options**: Accelerator **GPU T4** (or P100), **Internet on**.
4. **Save Version → Save & Run All (Commit)**. It runs in the background; you can close the browser.
5. When it finishes: **Output** tab → download `edsr_pop.zip`. Copy `edsr_best.pth` from it to
   `InfraVision/checkpoints/sr/edsr_best.pth` on the laptop.

**If the run stops early**: add this notebook's previous version output as an input and run again;
training continues from the last finished epoch.


In [ ]:
import shutil, subprocess
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
REPO, OUT = WORK / "InfraVision", WORK / "sr"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", "feature/rev1",
                    "https://github.com/elzasabu/InfraVision.git", str(REPO)], check=True)

# The unzipped POP folder, wherever Kaggle mounted it
found = [p.parents[2] for p in INPUT.rglob("train.json") if p.parent.name == "Annotations"]
assert found, "POP not found: Add Input -> your pop_raw dataset"
POP = found[0]

# Continue an earlier run if its output was added as an input
OUT.mkdir(parents=True, exist_ok=True)
previous = list(INPUT.rglob("last.pth"))
if previous and not (OUT / "last.pth").exists():
    shutil.copy(previous[0], OUT / "last.pth")
    print("continuing from", previous[0])

import torch
print("POP:", POP, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU")


In [ ]:
# No --amp on purpose: fp16 turned run 2's weights to NaN. EDSR is small; fp32 takes ~2-3 h on a T4.
# A failed "!" command does not stop the notebook, so check the exit code.
p = subprocess.Popen(["python", "-m", "infravision.train.sr", '--pop', str(POP), '--out', str(OUT), '--resume', '--workers', '4', '--epochs', '30', '--batch', '16'], cwd=REPO,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:   # live log in the cell and in Kaggle's Logs tab
    print(line, end="", flush=True)
assert p.wait() == 0, "training failed: see the error above"

In [ ]:
import json

m = json.loads((OUT / "metrics.json").read_text())
b = m["history"][0]["val_psnr_bicubic"]
for h in m["history"]:
    print(f"epoch {h['epoch']:3d}  PSNR {h['val_psnr']:.2f} dB (bicubic {b:.2f})  SSIM {h['val_ssim']:.4f}")
best = m["best"]
print("\nbest epoch", best["epoch"], "->", "BEATS bicubic" if best["val_psnr"] > b else "does NOT beat bicubic (keep sr.backend: bicubic)",
      f"by {best['val_psnr'] - b:+.2f} dB")


## Package the results (best weights + metrics; `last.pth` stays in the Output for resuming)

In [ ]:
pack = WORK / "sr_pack"
pack.mkdir(exist_ok=True)
for f in OUT.iterdir():
    if f.name != "last.pth":
        shutil.copy(f, pack / f.name)
print("Saved:", shutil.make_archive(str(WORK / "edsr_pop"), "zip", pack))
